# Day 1-7: Someone Already Built It

Every Friday from 1 to 4 PM, Mateo searches arXiv, saves the new papers to the lab folder, and writes the lab's digest. Three weeks ago he skimmed past the paper that scooped the lab's project.

He asked his agent to take over Fridays. It said **"I can't."**

Follow along with the slides. Stop at each **Try it**.

## 1. Add your API key

In the **Secrets** panel (the key icon), make sure `GOOGLE_API_KEY` exists and **Notebook access** is on.

In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
print("API key loaded.")

## 2. Install

`uv` gives you `uvx`, which downloads and launches a Python program in one step. Ready-made MCP servers are usually started this way.

In [ ]:
%pip install -q -U "langchain[mcp]>=1.4" langchain-google-genai uv

A Colab quirk: servers launched from a notebook need a real file to write their logs to. Run this once and move on.

In [ ]:
import sys, logging

sys.stderr = open("server_logs.txt", "a")
logging.getLogger("fastmcp").setLevel(logging.WARNING)

## 3. "I can't"

Mateo's request, sent to an agent with no tools.

In [ ]:
from langchain.agents import create_agent

MODEL = "google_genai:gemini-flash-lite-latest"
TOPIC = "LLM agents that automate scientific literature review"

FRIDAY = (
    f"Find this week's new arXiv papers on {TOPIC}. "
    "Pick the 3 most relevant, save them for the lab, and write a short digest."
)

agent = create_agent(MODEL, tools=[])
result = await agent.ainvoke({"messages": [{"role": "user", "content": FRIDAY}]})
print(result["messages"][-1].text)

The model knows what arXiv is. It knows how to write a digest. What's missing?

## 4. Plug in someone else's server

Someone already built an arXiv MCP server. You don't write it, and you don't download its code yourself: `uvx` fetches it and launches it **on this machine**.

In [ ]:
from langchain.mcp import MCPAdapter

config = {"mcpServers": {
    "arxiv": {"command": "uvx", "args": ["arxiv-mcp-server"]},   # launched here (stdio)
}}

async with MCPAdapter(config) as adapter:
    tools = await adapter.list_tools()

print(len(tools), "tools")
for t in tools:
    print(t.name)

How many lines of arXiv code did you write?

## 5. Mateo's Friday, again

Same request, same model. Now it has tools.

In [ ]:
from datetime import date

SYSTEM_PROMPT = (
    f"Today is {date.today()}. You are the research assistant for Mateo's lab. "
    "Use your tools; don't guess about papers."
)

agent = create_agent(MODEL, tools, system_prompt=SYSTEM_PROMPT)
result = await agent.ainvoke({"messages": [{"role": "user", "content": FRIDAY}]})

for m in result["messages"]:
    for call in getattr(m, "tool_calls", []):
        print("called:", call["name"])
print()
print(result["messages"][-1].text)

**Try it:** change `TOPIC` in section 3 to something you care about, then rerun sections 3 and 5.

## 6. Next Friday

Mateo goes to the lab folder to find last week's paper. Where did the agent save it?

In [ ]:
!mkdir -p lab_papers
print("Lab folder:")
!ls lab_papers
print()
print("Where the arXiv server put them:")
!ls ~/.arxiv-mcp-server/papers

The arXiv server saved the papers **for itself**, in a hidden folder. The lab's folder is empty.

What does the agent need now? Would someone already have built it?

## 7. A second server

Add the official filesystem server to the **same config**. The folders at the end of `args` are the only folders it's allowed to touch: the lab's folder, and the arXiv server's hidden one.

In [ ]:
ARXIV_FOLDER = "/root/.arxiv-mcp-server/papers"   # where the arXiv server saves papers
LAB_FOLDER = "/content/lab_papers"                # where the lab looks

config = {"mcpServers": {
    "arxiv": {"command": "uvx", "args": ["arxiv-mcp-server"]},
    "files": {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", LAB_FOLDER, ARXIV_FOLDER]},
}}

async with MCPAdapter(config) as adapter:
    tools = await adapter.list_tools()

print(len(tools), "tools")
for t in tools:
    print(t.name)

In [ ]:
FRIDAY = (
    f"Find this week's new arXiv papers on {TOPIC}. Pick the 3 most relevant and download them. "
    f"Move each downloaded paper's .md file from {ARXIV_FOLDER} into {LAB_FOLDER}. "
    f"Then save {LAB_FOLDER}/digest.md with each paper's title, arXiv link, and a 2-sentence summary."
)

agent = create_agent(MODEL, tools, system_prompt=SYSTEM_PROMPT)
result = await agent.ainvoke({"messages": [{"role": "user", "content": FRIDAY}]})

for m in result["messages"]:
    for call in getattr(m, "tool_calls", []):
        print("called:", call["name"])
print()
print(result["messages"][-1].text)

Open the **Files** pane (folder icon) and look in `lab_papers`. One agent, two servers, and you wrote neither.

In [ ]:
!ls lab_papers
!cat lab_papers/digest.md

## 8. Your turn

Head to the lesson's **Apply** section: give your Project 1 agent the docs server, a filesystem server, and one server you pick.